# 🔍 Data Exploration Playground
## Understanding the TCND dataset structure before modelling

**Purpose**: Load, inspect, and validate all data modalities to ensure correct data loading.

---

In [ ]:
import os, glob, math, time, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from netCDF4 import Dataset as NC4Dataset

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

PROJECT_ROOT = Path('/Users/thiruanand/MLfTCC')
DATA_ROOT    = PROJECT_ROOT / 'Notebooks' / 'data' / 'tropicyclonenet' / 'TCND_test'
BASINS = ['EP', 'NA', 'NI', 'SI', 'SP', 'WP']

print(f"Data root exists: {DATA_ROOT.exists()}")
print(f"Contents: {[x.name for x in DATA_ROOT.iterdir() if x.name != '.DS_Store']}")

## 1. Data1D — Raw File Inspection
Inspect the tab-separated `.txt` files to understand column order and meaning.

In [ ]:
# Pick a sample file
sample_basin = 'EP'
d1d_dir = DATA_ROOT / 'Data1D' / sample_basin / 'test'
txt_files = sorted(d1d_dir.glob('*.txt'))
print(f"Found {len(txt_files)} text files in {d1d_dir}")
print(f"First file: {txt_files[0].name}")

# Read raw lines
with open(txt_files[0]) as f:
    lines = f.readlines()
print(f"\nRaw lines (first 5):")
for line in lines[:5]:
    print(f"  {line.strip()}")

print(f"\nColumns appear to be tab-separated:")
cols = lines[0].strip().split('\t')
print(f"  {len(cols)} columns: {cols}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CRITICAL: Column order as defined in src/data/dataset.py (line 294-296)
#
# The CORRECT column order is:
#   ID | FLAG | LAT_norm | LONG_norm | WND_norm | PRES_norm | YYYYMMDDHH | Name
#
# ⚠️ The base notebook basin_generalisation.ipynb uses WRONG column names:
#   ['ID','col1','LONG_norm','LAT_norm','PRES_norm','WND_norm','datetime','name']
#   This swaps LAT↔LONG and WND↔PRES!
# ══════════════════════════════════════════════════════════════════════════════

CORRECT_COLS = ['ID', 'FLAG', 'LAT_norm', 'LONG_norm', 'WND_norm', 'PRES_norm', 'YYYYMMDDHH', 'Name']
WRONG_COLS   = ['ID', 'col1', 'LONG_norm', 'LAT_norm', 'PRES_norm', 'WND_norm', 'datetime', 'name']

df_correct = pd.read_csv(txt_files[0], sep='\t', header=None, names=CORRECT_COLS)
df_wrong   = pd.read_csv(txt_files[0], sep='\t', header=None, names=WRONG_COLS)

print("With CORRECT column order (src/data/dataset.py):")
display(df_correct.head())

print("\nWith WRONG column order (basin_generalisation.ipynb):")
display(df_wrong.head())

print("\n⚠️ Notice: The notebook swaps LAT with LONG and WND with PRES!")
print("Compare column 2 (0-indexed): should be LAT_norm, notebook calls it LONG_norm")

In [ ]:
# Inspect value ranges to confirm which interpretation is correct
print("Value ranges (using CORRECT column order):")
for col in ['LAT_norm', 'LONG_norm', 'WND_norm', 'PRES_norm']:
    vals = df_correct[col]
    print(f"  {col:15s}: min={vals.min():8.3f}, max={vals.max():8.3f}, mean={vals.mean():8.3f}")

print("\nExpected ranges (from src/data/dataset.py comments):")
print("  LAT_norm  = (LAT_raw + 900) / 1800  → roughly [0, 1] or unnormalized")
print("  LONG_norm = LONG_raw / 3600           → roughly [0, 1] or unnormalized")
print("  WND_norm  = WND_ms / 100              → roughly [0, 1]")
print("  PRES_norm = (PRES_hPa - 870) / 1080   → roughly [0, 1]")

print("\n→ Check if the value ranges match the expected meaning for each column.")

## 2. Env-Data — Raw `.npy` Inspection
Inspect the keys, types, and values in the environmental data dictionaries.

In [ ]:
# Load a sample Env-Data file
env_dir = DATA_ROOT / 'Env-Data' / sample_basin
npy_files = sorted(env_dir.rglob('*.npy'))[:5]
print(f"Found {len(list(env_dir.rglob('*.npy')))} .npy files in {env_dir}")
print(f"\nSample file: {npy_files[0]}")

env = np.load(npy_files[0], allow_pickle=True).item()
print(f"\nKeys ({len(env)}):")
for k, v in env.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:25s}: ndarray shape={v.shape}, dtype={v.dtype}")
        print(f"  {'':25s}  values={v}")
    elif isinstance(v, tuple):
        print(f"  {k:25s}: tuple = {v}")
    else:
        print(f"  {k:25s}: {type(v).__name__} = {v}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# KEY FINDING: The Env-Data has ACTUAL TARGET LABELS:
#   future_inte_change24  → intensity change class (0-4)
#   future_direction24    → direction class (0-7)
#
# These are the GROUND TRUTH labels provided by the dataset.
#
# ⚠️ basin_generalisation.ipynb IGNORES these and computes its own targets
#    using a 4-step-ahead wind difference, which is INCORRECT.
# ══════════════════════════════════════════════════════════════════════════════

print("TARGET LABELS in Env-Data:")
print(f"  future_inte_change24 = {env.get('future_inte_change24', 'MISSING')}")
print(f"  future_direction24   = {env.get('future_direction24', 'MISSING')}")

print("\nHISTORY features (inputs, not labels):")
print(f"  history_inte_change24 = {env.get('history_inte_change24', 'MISSING')}")
print(f"  history_direction12   = {env.get('history_direction12', 'MISSING')}")
print(f"  history_direction24   = {env.get('history_direction24', 'MISSING')}")

print("\n→ The src/data/dataset.py correctly uses these as targets (lines 528-529)")
print("→ The R&D notebook should do the same.")

In [ ]:
# Check target label distributions across multiple files
print("Scanning target labels across all basins...")
label_counts = {'intensity': Counter(), 'direction': Counter(), 'missing_int': 0, 'missing_dir': 0}

all_npy = list((DATA_ROOT / 'Env-Data').rglob('*.npy'))
print(f"Total .npy files: {len(all_npy)}")

for npy_path in all_npy:
    try:
        d = np.load(npy_path, allow_pickle=True).item()
        y_int = d.get('future_inte_change24', None)
        y_dir = d.get('future_direction24', None)
        if y_int is not None:
            val = int(np.asarray(y_int).ravel()[0])
            label_counts['intensity'][val] += 1
        else:
            label_counts['missing_int'] += 1
        if y_dir is not None:
            val = int(np.asarray(y_dir).ravel()[0])
            label_counts['direction'][val] += 1
        else:
            label_counts['missing_dir'] += 1
    except:
        pass

print(f"\nIntensity change class distribution (future_inte_change24):")
for cls in sorted(label_counts['intensity'].keys()):
    print(f"  Class {cls}: {label_counts['intensity'][cls]:5d}")
print(f"  Missing: {label_counts['missing_int']}")

print(f"\nDirection class distribution (future_direction24):")
for cls in sorted(label_counts['direction'].keys()):
    print(f"  Class {cls}: {label_counts['direction'][cls]:5d}")
print(f"  Missing: {label_counts['missing_dir']}")

## 3. Data3D — NetCDF Inspection

In [ ]:
# Load a sample Data3D file
nc_files = sorted((DATA_ROOT / 'Data3D' / sample_basin).rglob('*.nc'))[:3]
print(f"Found {len(list((DATA_ROOT / 'Data3D' / sample_basin).rglob('*.nc')))} NetCDF files")
print(f"\nSample: {nc_files[0]}")

with NC4Dataset(str(nc_files[0]), 'r') as nc:
    print(f"\nVariables:")
    for name, var in nc.variables.items():
        print(f"  {name:10s}: shape={var.shape}, dtype={var.dtype}")
    
    print(f"\nDimensions:")
    for name, dim in nc.dimensions.items():
        print(f"  {name}: size={len(dim)}")
    
    # Quick value ranges
    for vname in ['u', 'v', 'z', 'sst']:
        data = np.array(nc.variables[vname][:])
        valid = data[~np.isnan(data) & (data < 1e30)]
        if len(valid) > 0:
            print(f"  {vname}: valid_range=[{valid.min():.2f}, {valid.max():.2f}], mean={valid.mean():.2f}")

## 4. Cross-Modality Alignment Check
Verify that Data1D, Data3D, and Env-Data timestamps align for the same storm.

In [ ]:
# Check alignment for one storm
storm_file = txt_files[0]  # First EP storm
storm_name = storm_file.stem  # e.g. EP2017BSTDORA
basin = 'EP'
year = storm_name[2:6]
tc_name = storm_name[9:]  # After 'EP2017BST'

df = pd.read_csv(storm_file, sep='\t', header=None, names=CORRECT_COLS)
timestamps = [str(int(ts)) for ts in df['YYYYMMDDHH']]

print(f"Storm: {storm_name} ({tc_name}, {year})")
print(f"Data1D timesteps: {len(timestamps)}")

# Check which timesteps have matching Data3D and Env-Data
matched, missing_3d, missing_env = 0, 0, 0
for ts in timestamps:
    has_3d = (DATA_ROOT / 'Data3D' / basin / year / tc_name / f'TCND_{tc_name}_{ts}_sst_z_u_v.nc').exists()
    has_env = (DATA_ROOT / 'Env-Data' / basin / year / tc_name / f'{ts}.npy').exists()
    if has_3d and has_env:
        matched += 1
    if not has_3d:
        missing_3d += 1
    if not has_env:
        missing_env += 1

print(f"Matched (all 3 modalities): {matched}/{len(timestamps)}")
print(f"Missing Data3D: {missing_3d}")
print(f"Missing Env-Data: {missing_env}")

## 5. Summary of Findings

In [ ]:
print("═" * 80)
print("DATA EXPLORATION FINDINGS")
print("═" * 80)

print("""
1. DATA1D COLUMN ORDER:
   ✅ Correct: ID | FLAG | LAT_norm | LONG_norm | WND_norm | PRES_norm | YYYYMMDDHH | Name
   ❌ basin_generalisation.ipynb uses: ID | col1 | LONG_norm | LAT_norm | PRES_norm | WND_norm | datetime | name
   → LAT and LONG are SWAPPED
   → WND and PRES are SWAPPED

2. TARGET LABELS:
   ✅ Correct: Read from Env-Data .npy files:
      - future_inte_change24 → y_intensity (0-4, with -1 = unknown)
      - future_direction24   → y_direction (0-7, with -1 = unknown)
   ❌ basin_generalisation.ipynb computes targets from 4-step-ahead wind difference
      This is an approximation, not the actual ground truth.

3. ENV-DATA STRUCTURE:
   Feature keys (input): month, area, intensity_class, wind, move_velocity,
                          location_long, location_lat, history_direction12/24,
                          history_inte_change24, location
   Label keys (target):  future_inte_change24, future_direction24
   Total input dim: 77 (12+6+6+1+1+36+12+1+1+1)

4. DATA3D STRUCTURE:
   Variables: u(1,4,81,81), v(1,4,81,81), z(1,4,81,81), sst(1,81,81) → 13 channels
   4 pressure levels for u/v/z; SST has fill values ~9.97e36 for land

ACTION REQUIRED:
   All R&D notebooks (basin_generalisation.ipynb + 3 new ones) need to be
   updated to:
   a) Use correct Data1D column order
   b) Read targets from Env-Data instead of computing them
""")

print("✅ Data exploration complete!")